# 04C — Compare saved baseline or a named eight-configuration experiment

Read-only: no feature generation, training or repeated model TEST evaluation.
Leave `EXPERIMENT_NAME` empty to view the immutable accepted baseline. Enter the name
used in 04D to view its completed eight-configuration group on Drive.
The provisional shortlist uses selected-validation macro-F1, then parameter count and run ID;
TEST metrics are descriptive, not tuning signals. Storage uses `tools/project_paths.py`.


In [ ]:
# Each notebook kernel discovers its own runtime checkout.
from pathlib import Path, PureWindowsPath
import os, subprocess, sys
_repo_hint = os.environ.get("EDGEAI_REPO_ROOT")
if _repo_hint and os.name != "nt" and PureWindowsPath(_repo_hint).drive:
    raise RuntimeError("EDGEAI_REPO_ROOT must name the checkout in this runtime, not a Windows drive")
_start = Path(_repo_hint).expanduser().resolve() if _repo_hint else Path.cwd().resolve()
_candidates = [_start, *_start.parents, Path("/content/edge-ai")]
_repo = next((p for p in _candidates if (p / "tools/project_paths.py").is_file()), None)
if _repo is None:
    try:
        import google.colab
    except ImportError:
        raise RuntimeError("Set EDGEAI_REPO_ROOT to your local checkout and rerun this cell") from None
    _repo = Path(_repo_hint or "/content/edge-ai").expanduser().resolve()
    if _repo.exists():
        raise RuntimeError(f"{_repo} exists but is not an EdgeAI checkout; set EDGEAI_REPO_ROOT to the correct checkout")
    subprocess.run(["git", "clone", "https://github.com/ancaranovik/Mosquito_Wingbeat_v1.git", str(_repo)], check=True)
if not (_repo / "tools/notebook_runtime.py").is_file():
    raise RuntimeError("Checkout is missing notebook_runtime.py. Pull the latest GitHub commit in this runtime, then rerun this cell")
sys.path.insert(0, str(_repo / "tools"))
from notebook_runtime import initialize
PATHS = initialize(_repo)
from project_paths import REPO_ROOT, ProjectPaths, resolve_path, logical_relative
from pathlib import Path
import sys
sys.dont_write_bytecode = True
PROJECT = REPO_ROOT
sys.path.insert(0, str(PROJECT / "tools"))
import importlib
import stage04_data, model_zoo
for module in (stage04_data, model_zoo):
    importlib.reload(module)
import pandas as pd
from IPython.display import display
from model_zoo import FAMILIES, architecture_config, build_model
from stage04_data import BRANCHES, CLASSES, ROOT, prepare_caches, read_json


In [ ]:
# Để trống: baseline cố định. Nhập tên đã dùng trong 04D: kết quả experiment trên Drive.
EXPERIMENT_NAME = ""

all_complete = False  # Prevent plots from reusing a previous selection after a failed check.
RUNS = None
result_root = (PATHS.experiment(EXPERIMENT_NAME) / "results"
               if EXPERIMENT_NAME else PATHS.BASELINE_ROOT)
print("Đang đọc:", result_root)
status = []
for frontend in BRANCHES:
    for family in FAMILIES:
        path = result_root / "runs" / f"{frontend}_{family}_seed42" / "result.json"
        status.append({"Frontend": frontend, "Model": family,
                       "Status": read_json(path)["status"] if path.is_file() else "NOT RUN"})
display(pd.DataFrame(status))
if EXPERIMENT_NAME:
    metadata = read_json(result_root.parent / "config/metadata.json")
    print("Experiment status:", metadata["status"],
          f'{metadata["completed_runs"]}/{metadata["expected_runs"]}')
    display(pd.Series({"experiment_id": metadata["experiment_id"],
                       "git_commit": metadata["git_commit"],
                       "description": metadata["configuration"]["description"],
                       "configuration": metadata["configuration"],
                       "runtime": metadata["runtime"]}, name="Run provenance"))
    from experiment_review import verify_experiment, comparison_summary
    records = verify_experiment(EXPERIMENT_NAME)
    if len(records) != 8:
        raise RuntimeError("04C requires a complete eight-configuration experiment; use the full-suite template in 04D.")
    summary, comparison = comparison_summary(records)
else:
    from baseline_review import verify_baseline
    import experiment_training
    records = verify_baseline()
    comparison = pd.DataFrame([experiment_training.comparison_row(r) for r in records])
    summary = read_json(result_root / "comparison.json")
RUNS = result_root / "runs"
all_complete = True
print("Saved training protocol/parameters:", read_json(result_root / "protocol.json")["training_configuration"])
display(comparison)
print("Provisional Stage-05 shortlist (validation only):", summary["stage05_shortlist"])


## Saved training histories and confusion matrices

TRAIN accuracy/F1 are online metrics with dropout; validation uses evaluation mode. All four classes
remain in macro metrics even when predictions for a class are absent. Confusion matrices use true rows
and predicted columns. Source-level metrics are secondary and retain one observation per source name.


In [ ]:
if all_complete:
    import matplotlib.pyplot as plt
    for frontend in BRANCHES:
        for family in FAMILIES:
            directory = RUNS / f"{frontend}_{family}_seed42"
            result = read_json(directory / "result.json")
            history = read_json(directory / "history.json")
            fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
            epochs = [h["epoch"] for h in history]
            for split in ("train", "validation"):
                axes[0].plot(epochs, [h[split]["loss"] for h in history], label=split)
                axes[1].plot(epochs, [h[split]["macro_f1"] for h in history], label=split)
            axes[0].set(title="Weighted loss", xlabel="Epoch"); axes[0].legend()
            axes[1].set(title="Macro-F1", xlabel="Epoch"); axes[1].legend()
            cm = result["final_test_metrics"]["confusion_matrix"]
            axes[2].imshow(cm, cmap="Blues")
            for i in range(4):
                for j in range(4):
                    axes[2].text(j, i, cm[i][j], ha="center", va="center")
            axes[2].set(title="TEST example confusion", xlabel="Predicted label", ylabel="True label",
                        xticks=range(4), yticks=range(4))
            fig.suptitle(f"{frontend} / {family}; selected epoch {result['best_epoch']}")
            fig.tight_layout(); plt.show()
            display(pd.DataFrame(result["final_test_metrics"]["per_class"]).T)
            display(pd.Series(result["source_test_metrics"], name="Secondary source evaluation"))


## Limits before Stage 05

No Edge winner is declared. Float32 checkpoint size is not INT8 size, RAM, Flash or latency.
This one-seed comparison makes no claims of superiority to published literature. Multiple windows may
share one recording; source-name isolation does not prove biological/session/domain independence.
The frozen acquisition-method confounding remains. Read [Stage 04 methodology](../../docs/STAGE04.md).
